## GPT prompting: n20 examples, first filter (uses prompt v02)

### requires python >= 3.10

In [1]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [2]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [3]:
from prompts.semantic_categories.v02.prompt import SYSTEM_PROMPT, FEW_SHOTS_STR, FEW_SHOTS

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [6]:
RESULTS_DIR = "../../results/"

# andmefail
EXAMPLE_FILE = "../../data/n20_examples_large_v01.csv"
# filtri tulemusfail
GPT_ANSWER_FILE = RESULTS_DIR + "n20_examples_large_v01/gpt_v01/"+ "gpt_b10_run01.csv"

CONF_FILE = "../../../../v04_verb-case_pattern/minu_code/azure.ini"


# OSA I : Andmed

In [34]:
df1 = pd.read_csv(EXAMPLE_FILE, encoding="utf-8",  sep=",")

spatial_obl_ex = df1.iloc[:10000]
spatial_obl_ex = spatial_obl_ex.sample(frac=1)

## valida üks meetod: kas välja jätta juba testitud 10K näidet või võtta kõik
# kommenteeri välja see variant, mida ei kasuta

# 1) jätta välja esimesed 10K
#spatial_obl_ex = df.iloc[10000:]
#spatial_obl_ex = spatial_obl_ex.sample(frac=1)

# 2) kui ei jäta välja siis see:
# + shuffle (frac)
# enne/pärast sample ka .iloc[x:n] kui vaja võtta subset
#spatial_obl_ex = df.sample(frac=1)


In [35]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence,sentence_id,timex_tag,ekilex_tag,ner_tag
1377,11432411,korral,kord,tasuma,NaN,ad,"“ Ei tea , kas esimesel korral tasub lõpuni minna , ” kahtlesin .",7108634,NaN,NaN,NaN
5095,4175718,juhatusel,juhatus,pakkuma,NaN,ad,"Eesti Filharmoonia Kammerkoor ja Tallinna Kammerorkester pakuvad täna õhtul Tõnu Kaljuste juhatusel Tallinna Metodisti kirikus toimuval kontserdil valiku teostest , mida koor ja orkester esitavad reedel algaval kolmenädalasel USA ja Kanada turneel .",2601827,NaN,NaN,NaN
6695,2670999,patsiendile,patsient,seletama,NaN,all,"Kullamaa seletab patsiendile alati , et iga protees on organismile võõrkeha ja ümber selle tekitab organism poole aasta jooksul sidekoelise kapsli .",1677278,NaN,NaN,NaN
5365,1196732,reisijale,reisija,virutama,NaN,all,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",750912,NaN,alive,NaN
3942,7604499,poolajal,poolaeg,lülitama,NaN,ad,""" Seda ma paraku ei tea , sest lülitasin teleka sisse alles teisel poolajal . """,4733683,NaN,time,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3632,15722082,päevil,päev,käsitlema,NaN,ad,Film käsitleb sündmusi 1943. aastal Varssavis kannatusnädala päevil ja põhineb Jerzy Andrzejewski romaanil .,9799023,NaN,NaN,NaN
2579,7865678,seanssidest,seanss,jääma,kõrvale,el,"Kui aga tegemist on abikaasade omavaheliste probleemidega , jäävad lapsed reeglina neist seanssidest kõrvale .",4898982,NaN,NaN,NaN
8936,20625029,esitamisel,esitamine,saama,sisse,ad,Riigikogulased saavad töötõendi esitamisel tasuta sisse .,12893712,NaN,NaN,NaN
9335,824406,Järvel,järv,võitma,NaN,ad,"Üksikpiltidest võitis Hermes Sarapuu "" Järvel "" .",523151,NaN,NaN,LOC


# OSA II : GPT

## GPT jaoks vajalik

In [9]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [10]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

In [11]:
SYSTEM_PROMPT

'\nYou are a classification assistant.\nIn this task location refers to "adverbial of place" (Estonian: kohamäärus) or "locative adverb".\nYour task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" functions as a location in the context of the sentence.\nAdverbial of place answers to the question “where” (kus?/kuhu?/kust?) in the context of the sentence.\nIt is a place or concept where something or someone is located, goes to or comes from.\nCriteria:\n- concrete place (bank, table, Berlin)\n- abstract (literature, soul, TV channels, government, top of a group, history, thought, domain)\n- inanimate (journal, chair, wifi, bag, medal, toy, food, computer, wire, body parts)\n- alive (mother, Peter, dog, doctor, teacher)\n- event (dress rehearsal, camp, class, situation, meeting)\n- state or condition conceptualized as space (life, trouble, consciousness, attitude)\n- Locations ARE NOT phrases that show time, state of being, owner, experience

In [6]:
#FEW_SHOTS_STR

## Andmete söötmine

In [36]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    # attempt on selleks kui ei saa õiget arvu vastuseid tagasi, siis kui palju kordi uuesti proovida
    # elu näitas, et batch 12 või suurema puhul uuesti proovimine ei anna õiget arvu vastuseid
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":   json.dumps(user_payload, ensure_ascii=False) }
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [37]:
# kuna enamus vastuseid peaks olema "no" siis küsime "yes" puhul põhjendust
def explain_locations(
    client, 
    deployment,
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    subset_ratio: float = 0.0,

) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset (sanity check for "no" answers)
    diag_count = int(len(no_indices) * subset_ratio)
    diag_indices = no_indices[:diag_count]

    explain_indices = yes_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as adverbial of place ('yes') or not adverbial of place ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, without markdown and code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" +  json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]
    #return None, None,None 
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices


## NB! muuda max_allowed_tok kui vaja

In [38]:
df = spatial_obl_ex

In [40]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10

# kui suure osa võtta "yes" vastustest "why" küsimusse
# kui on 0.2, siis võiks max 2/10 "yes" olla põhjendatud, kui on juba 1 "no" siis on ainult 1 "yes" põhjendatud
no_subset_ratio = 0.2
batch_start_index = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 4500000


batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, FEW_SHOTS_STR, SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_locations(
            batch=batch,
            yes_no_results=result_yesno,
            subset_ratio=no_subset_ratio,
            client=client, 
            deployment=DEPLOYMENT
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    




1000it [43:29,  2.61s/it]


In [41]:
used_tokens # 10 lauset, batch 10 -> u 3700 tokenit, 10K lauset b10 -> 3,986,441 tokenit

3986441

In [42]:
len(results)

10000

## andmed tabelisse 

### enne kontroll kas andmeid on puudu ja vastavad lüngad täita

In [43]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification"] = new_results
    df["explanation"] = new_explanations

    #df["explanation"] = new_explanations 

In [44]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence,sentence_id,timex_tag,ekilex_tag,ner_tag,classification,explanation
1377,11432411,korral,kord,tasuma,NaN,ad,"“ Ei tea , kas esimesel korral tasub lõpuni minna , ” kahtlesin .",7108634,NaN,NaN,NaN,no,"The phrase 'korral' refers to 'occasion' and does not indicate a location, hence it is not adverbial of place."
5095,4175718,juhatusel,juhatus,pakkuma,NaN,ad,"Eesti Filharmoonia Kammerkoor ja Tallinna Kammerorkester pakuvad täna õhtul Tõnu Kaljuste juhatusel Tallinna Metodisti kirikus toimuval kontserdil valiku teostest , mida koor ja orkester esitavad reedel algaval kolmenädalasel USA ja Kanada turneel .",2601827,NaN,NaN,NaN,no,"The phrase 'juhatusel' refers to 'under the direction' and is describing leadership or guidance, not location, so it is not adverbial of place."
6695,2670999,patsiendile,patsient,seletama,NaN,all,"Kullamaa seletab patsiendile alati , et iga protees on organismile võõrkeha ja ümber selle tekitab organism poole aasta jooksul sidekoelise kapsli .",1677278,NaN,NaN,NaN,no,
5365,1196732,reisijale,reisija,virutama,NaN,all,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",750912,NaN,alive,NaN,no,
3942,7604499,poolajal,poolaeg,lülitama,NaN,ad,""" Seda ma paraku ei tea , sest lülitasin teleka sisse alles teisel poolajal . """,4733683,NaN,time,NaN,no,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3632,15722082,päevil,päev,käsitlema,NaN,ad,Film käsitleb sündmusi 1943. aastal Varssavis kannatusnädala päevil ja põhineb Jerzy Andrzejewski romaanil .,9799023,NaN,NaN,NaN,no,
2579,7865678,seanssidest,seanss,jääma,kõrvale,el,"Kui aga tegemist on abikaasade omavaheliste probleemidega , jäävad lapsed reeglina neist seanssidest kõrvale .",4898982,NaN,NaN,NaN,yes,"The phrase 'seanssidest' describes a location or context from which someone is excluded, thus classified as adverbial of place."
8936,20625029,esitamisel,esitamine,saama,sisse,ad,Riigikogulased saavad töötõendi esitamisel tasuta sisse .,12893712,NaN,NaN,NaN,no,
9335,824406,Järvel,järv,võitma,NaN,ad,"Üksikpiltidest võitis Hermes Sarapuu "" Järvel "" .",523151,NaN,NaN,LOC,yes,"The phrase 'Järvel' (on the lake) specifies a location, making it adverbial of place."


### salvestada tulemused faili

In [45]:
df.to_csv(GPT_ANSWER_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## optional saving

fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)

In [46]:
df = pd.read_csv(GPT_ANSWER_FILE , encoding="utf-8",  sep=",")

In [47]:
print("yes:", len(df[df["classification"]=="yes"])) 
print("no explained:", len(df[(df["classification"]=="no") & (~df["explanation"].isna())]))
print("no:", len(df[df["classification"]=="no"])) 

yes: 4353
no explained: 773
no: 5647
